# NB9b - 5-fold cross-validation for the baselines (B1, B2, B3)

**Run on:** GPU, Internet ON. **Input:** `eurosat-geograph-nb1-v2` only (new version).
Same protocol and recipe as NB9a / NB2: fold k = test, (k+1) mod 5 = validation, the other three = training, normalisation from the folds other than k. Saves `nb9b_cv_results.csv` and `nb9b_cv_probs.npz` after every run.
**Quick test first:** `CV_FOLDS=[0]`, `CV_CODES=["B2"]`, `EPOCHS_CV=2`.


## Cell 1 - Configuration, seeding, GPU check, locate and load the NB1 data

In [1]:
import os, glob, json, time, random, subprocess, sys
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

# ---------------- one recipe for ALL experiments (keep identical in NB4/NB5) ----------------
CFG = dict(seed=42, epochs=30, batch_size=128, lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
           drop_rate=0.2, img_size=64, swin_window=4, num_workers=2,
           backbone_cnn="tf_efficientnetv2_s.in21k_ft_in1k", backbone_vit="swin_tiny_patch4_window7_224")
RUN_CV = True          # True: also run the 5-fold cross-validation (Cell 7); heavy
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
      "| torch", torch.__version__, "| timm", timm.__version__)
if DEVICE.type != "cuda":
    print("WARNING: no GPU found, training will be very slow. Set Accelerator to GPU.")

cand = sorted(glob.glob("/kaggle/input/**/X16_float16.npy", recursive=True) + glob.glob(f"{WORK}/**/X16_float16.npy", recursive=True))
assert cand, "X16_float16.npy not found: add the NB1 output dataset under Input"
assert len(cand) == 1, f"Several X16_float16.npy in Input (old NB1 attached?): {cand}"
NB1_DIR = os.path.dirname(cand[0])
print("NB1_DIR =", NB1_DIR, "|", sorted(os.listdir(NB1_DIR)))

idx = pd.read_csv(f"{NB1_DIR}/folds.csv")
stats = json.load(open(f"{NB1_DIR}/channel_stats.json"))
y_all = np.load(f"{NB1_DIR}/labels.npy")
try:
    X16 = np.load(f"{NB1_DIR}/X16_float16.npy")                  # ~3.5 GB in RAM: much faster than a memmap
except MemoryError:
    X16 = np.load(f"{NB1_DIR}/X16_float16.npy", mmap_mode="r")
    print("loaded as memmap (not enough RAM)")

assert X16.shape == (len(idx), 16, 64, 64), X16.shape
_nd = stats["phase1_train_folds_2_3_4"]["mean"][15]
print("NDBI train mean: %.4f (fixed NB1 about -0.18)" % _nd)
assert _nd > -0.25, "OLD NB1 stats (B12-based NDBI). Attach only the fixed NB1 output."
assert (idx["label"].values == y_all).all(), "labels.npy and folds.csv disagree"
CLASSES = [idx.loc[idx["label"] == i, "class"].iloc[0] for i in range(10)]
print("X16:", X16.shape, X16.dtype, "| classes:", CLASSES)
print("Phase 1 split counts:", idx["phase1"].value_counts().to_dict())

Device: cuda | GPU: Tesla T4 | torch 2.11.0+cu130 | timm 1.0.30
NB1_DIR = /kaggle/input/datasets/sonalidas6555/eurosat-nb1-output | ['X13_uint16.npy', 'X16_float16.npy', 'channel_stats.json', 'data_root.txt', 'eda_band_means.png', 'folds.csv', 'labels.npy', 'near_duplicates.csv']
NDBI train mean: -0.1765 (fixed NB1 about -0.18)
X16: (27000, 16, 64, 64) float16 | classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Phase 1 split counts: {'train': 16200, 'val': 5400, 'test': 5400}


## Cell 2 - Splits, Dataset/DataLoaders, metrics (Acc, Macro-F1, Kappa, ECE) and class-pair audit

In [2]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score, confusion_matrix)

FOLDS = idx["fold"].values

def make_split(mode="phase1", k=0):
    # phase1: test fold 0, val fold 1, train folds 2-4.  cv: test fold k, val fold (k+1)%5, train = the other three.
    if mode == "phase1":
        tr = np.where(np.isin(FOLDS, [2, 3, 4]))[0]; va = np.where(FOLDS == 1)[0]; te = np.where(FOLDS == 0)[0]
        st = stats["phase1_train_folds_2_3_4"]
        assert (idx["phase1"].values[tr] == "train").all() and (idx["phase1"].values[va] == "val").all() \
               and (idx["phase1"].values[te] == "test").all(), "split disagrees with folds.csv"
    else:
        te_f, va_f = k, (k + 1) % 5
        tr = np.where(~np.isin(FOLDS, [te_f, va_f]))[0]; va = np.where(FOLDS == va_f)[0]; te = np.where(FOLDS == te_f)[0]
        st = stats["cv_train_excluding_fold"][str(k)]          # statistics from folds other than the test fold
    assert len(set(tr) & set(te)) == 0 and len(set(va) & set(te)) == 0 and len(set(tr) & set(va)) == 0
    return dict(mode=mode, k=k, tr=tr, va=va, te=te,
                mu=np.array(st["mean"], np.float32), sd=np.array(st["std"], np.float32))

class EuroSATBaselineDataset(Dataset):
    def __init__(self, indices, channels, mu, sd, augment=False):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.channels = list(channels)
        self.mu = mu[self.channels][:, None, None]
        self.sd = sd[self.channels][:, None, None]
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        r = self.indices[i]
        x = np.array(X16[r, self.channels], dtype=np.float32)
        x = (x - self.mu) / (self.sd + 1e-6)
        if self.augment:                                          # 90-degree rotations and flips (no label change)
            k = random.randint(0, 3)
            if k: x = np.rot90(x, k, axes=(1, 2))
            if random.random() < 0.5: x = np.flip(x, axis=2)
            if random.random() < 0.5: x = np.flip(x, axis=1)
            x = np.ascontiguousarray(x)
        return torch.from_numpy(x), torch.tensor(y_all[r], dtype=torch.long)

def _winit(wid):
    s = (torch.initial_seed() + wid) % (2 ** 32)
    random.seed(s); np.random.seed(s)

def make_loaders(channels, split):
    bs, nw = CFG["batch_size"], CFG["num_workers"]
    g = torch.Generator(); g.manual_seed(CFG["seed"])
    kw = dict(num_workers=nw, pin_memory=True, worker_init_fn=_winit, persistent_workers=(nw > 0))
    mk = lambda ids, aug: EuroSATBaselineDataset(ids, channels, split["mu"], split["sd"], aug)
    tr = DataLoader(mk(split["tr"], True), batch_size=bs, shuffle=True, generator=g, drop_last=False, **kw)
    va = DataLoader(mk(split["va"], False), batch_size=bs * 2, shuffle=False, **kw)
    te = DataLoader(mk(split["te"], False), batch_size=bs * 2, shuffle=False, **kw)
    return tr, va, te

def compute_ece(probs, labels, n_bins=15):
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    acc = (pred == labels).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for b0, b1 in zip(bins[:-1], bins[1:]):
        m = (conf > b0) & (conf <= b1)
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def evaluate_predictions(probs, labels):
    p = probs.argmax(axis=1)
    return {"Acc": round(accuracy_score(labels, p) * 100, 3),
            "Macro_P": round(precision_score(labels, p, average="macro", zero_division=0) * 100, 3),
            "Macro_R": round(recall_score(labels, p, average="macro", zero_division=0) * 100, 3),
            "Macro_F1": round(f1_score(labels, p, average="macro", zero_division=0) * 100, 3),
            "Kappa": round(cohen_kappa_score(labels, p), 4),
            "ECE": round(compute_ece(probs, labels), 4)}

PAIRS = [("AnnualCrop", "PermanentCrop"), ("Residential", "Industrial"),
         ("HerbaceousVegetation", "Pasture"), ("River", "Highway")]

def class_pair_errors(cm, classes):
    tot_err = int(cm.sum() - np.trace(cm))
    rows = []
    for a, b in PAIRS:
        i, j = classes.index(a), classes.index(b)
        t = int(cm[i, j] + cm[j, i])
        rows.append({"pair": f"{a} <-> {b}", "a_as_b": int(cm[i, j]), "b_as_a": int(cm[j, i]),
                     "total_confusions": t, "share_of_all_errors_pct": round(100.0 * t / max(tot_err, 1), 2)})
    return pd.DataFrame(rows)

SPLIT = make_split("phase1")
print({k: len(SPLIT[k]) for k in ("tr", "va", "te")}, "| split OK, no overlap")
print("DataLoader, metric and audit functions ready.")

{'tr': 16200, 'va': 5400, 'te': 5400} | split OK, no overlap
DataLoader, metric and audit functions ready.


## Cell 3 - Models, trainer, profiling

- Pretrained weights are used when they can be downloaded; the result row records `Pretrained`.
- For 16-channel input, timm repeats and rescales the pretrained RGB first-layer filters, so only the first layer is RGB-initialised (state this in the paper).
- `img_size` in `CFG` is applied to all models (inputs are resized on the GPU when it is not 64).
- Latency: pure forward pass on random data, batch 128, after warm-up (no data loading). GFLOPs count a multiply-add as 2 FLOPs.

In [3]:
MODEL_SPECS = {
    "B1": dict(name="EfficientNetV2-S (RGB)",    channels=[3, 2, 1],        arch="cnn", ckpt="best_B1_rgb_cnn.pt"),
    "B2": dict(name="EfficientNetV2-S (16-Ch)",  channels=list(range(16)),  arch="cnn", ckpt="best_B2_16ch_cnn.pt"),
    "B3": dict(name="Swin-Tiny (16-Ch)",         channels=list(range(16)),  arch="vit", ckpt="best_B3_16ch_vit.pt"),
}
IMG = CFG["img_size"]

def prep(x):
    return x if IMG == 64 else F.interpolate(x, size=(IMG, IMG), mode="bilinear", align_corners=False)

def create(name, **kw):
    try:
        return timm.create_model(name, pretrained=True, **kw), True
    except Exception as e:
        print(f"WARNING: pretrained weights unavailable ({type(e).__name__}) -> training from scratch. "
              f"Turn Internet ON for pretrained weights.")
        return timm.create_model(name, pretrained=False, **kw), False

def build_model(code):
    spec = MODEL_SPECS[code]; n_ch = len(spec["channels"])
    if spec["arch"] == "cnn":
        return create(CFG["backbone_cnn"], in_chans=n_ch, num_classes=10, drop_rate=CFG["drop_rate"])
    return create(CFG["backbone_vit"], in_chans=n_ch, num_classes=10, drop_rate=CFG["drop_rate"],
                  img_size=IMG, window_size=CFG["swin_window"])

def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        x = prep(x.to(DEVICE, non_blocking=True))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            logits = model(x)
        P.append(torch.softmax(logits.float(), dim=-1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P, 0), np.concatenate(Y, 0)

@torch.no_grad()
def profile_model(model, n_ch, bs=128, reps=10):
    model.eval()
    gflops = float("nan")
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc:
            model(prep(torch.randn(1, n_ch, 64, 64, device=DEVICE)))
        gflops = fc.get_total_flops() / 1e9
    except Exception as e:
        print("FLOP counting unavailable:", type(e).__name__)
    x = prep(torch.randn(bs, n_ch, 64, 64, device=DEVICE))
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
        for _ in range(5): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(reps): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return gflops, (time.time() - t) * 1000.0 / (bs * reps)

def run_experiment(code, split, epochs=None, tag=""):
    epochs = epochs or CFG["epochs"]
    spec = MODEL_SPECS[code]; channels = spec["channels"]
    seed_all(CFG["seed"])
    tr_dl, va_dl, te_dl = make_loaders(channels, split)
    model, pretrained = build_model(code)
    model = model.to(DEVICE)
    params_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    crit = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    use_amp = DEVICE.type == "cuda"
    scaler = make_scaler(use_amp)

    ckpt = f"{WORK}/{spec['ckpt'].replace('.pt', tag + '.pt')}"
    best, best_ep, hist, t0 = -1.0, 0, [], time.time()
    for ep in range(1, epochs + 1):
        model.train(); tl = 0.0
        for x, y in tr_dl:
            x, y = prep(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y)
        sched.step()
        vp, vy = predict_loader(model, va_dl)
        vres = evaluate_predictions(vp, vy)
        hist.append({"epoch": ep, "train_loss": tl / len(tr_dl.dataset), "val_acc": vres["Acc"],
                     "val_macro_f1": vres["Macro_F1"], "val_ece": vres["ECE"]})
        if vres["Macro_F1"] > best:
            best, best_ep = vres["Macro_F1"], ep
            torch.save(model.state_dict(), ckpt)
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[{code}{tag}] Ep {ep:02d}/{epochs} | loss {tl/len(tr_dl.dataset):.4f} | "
                  f"val F1 {vres['Macro_F1']:.2f}% | best {best:.2f}% (ep {best_ep}) | {(time.time()-t0)/60:.1f} min")
    train_min = (time.time() - t0) / 60.0

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    te_p, te_y = predict_loader(model, te_dl)                     # test fold used once, with the best checkpoint
    va_p, va_y = predict_loader(model, va_dl)
    res = evaluate_predictions(te_p, te_y)
    gflops, ms = profile_model(model, len(channels))
    res.update({"Exp": code, "Model": spec["name"], "Channels": len(channels), "Params_M": round(params_m, 2),
                "GFLOPs": round(gflops, 3), "Infer_ms": round(ms, 4), "Train_Min": round(train_min, 2),
                "Best_Epoch": best_ep, "Pretrained": pretrained})
    print(f"--> {code} {tag} TEST: Acc={res['Acc']}% | Macro-F1={res['Macro_F1']}% | Kappa={res['Kappa']} | ECE={res['ECE']}\n")
    del model, opt
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res, te_p, te_y, va_p, va_y, pd.DataFrame(hist)

## Cell 4 - Run the cross-validation


In [4]:
CV_FOLDS = [0, 1, 2, 3, 4]
CV_CODES = ["B1", "B2", "B3"]            # B1 can be dropped if time is short
EPOCHS_CV = CFG["epochs"]                 # keep 30; use 2 only for the quick test
cv_rows, cv_probs = [], {}
for k in CV_FOLDS:
    sp = make_split("cv", k)
    print(f"=== fold {k}: train {len(sp['tr'])} | val {len(sp['va'])} | test {len(sp['te'])} ===")
    for code in CV_CODES:
        res, tp, ty, vp, vy, h = run_experiment(code, sp, epochs=EPOCHS_CV, tag=f"_f{k}")
        res["Fold"] = k; cv_rows.append(res)
        cv_probs[f"test_{code}_f{k}"] = tp; cv_probs[f"val_{code}_f{k}"] = vp; cv_probs[f"y_test_f{k}"] = ty; cv_probs[f"y_val_f{k}"] = vy
        pd.DataFrame(cv_rows).to_csv(f"{WORK}/nb9b_cv_results.csv", index=False)
        np.savez_compressed(f"{WORK}/nb9b_cv_probs.npz", **cv_probs)
        for p in glob.glob(f"{WORK}/best_{code}*_f{k}.pt"): os.remove(p)
cv_df = pd.DataFrame(cv_rows)
display(cv_df[["Exp", "Fold", "Acc", "Macro_F1", "Kappa", "ECE", "Best_Epoch", "Train_Min"]])
display(cv_df.groupby("Exp")[["Acc", "Macro_F1", "Kappa", "ECE"]].agg(["mean", "std"]).round(3))


=== fold 0: train 16200 | val 5400 | test 5400 ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.5MB            

model.safetensors: downloading bytes:           |  0.00B            

[B1_f0] Ep 01/30 | loss 2.4349 | val F1 89.85% | best 89.85% (ep 1) | 0.4 min
[B1_f0] Ep 05/30 | loss 0.6253 | val F1 97.15% | best 97.15% (ep 5) | 1.8 min
[B1_f0] Ep 10/30 | loss 0.5605 | val F1 98.07% | best 98.07% (ep 10) | 3.6 min
[B1_f0] Ep 15/30 | loss 0.5371 | val F1 98.30% | best 98.30% (ep 15) | 5.4 min
[B1_f0] Ep 20/30 | loss 0.5270 | val F1 98.27% | best 98.37% (ep 16) | 7.2 min
[B1_f0] Ep 25/30 | loss 0.5223 | val F1 98.41% | best 98.47% (ep 24) | 9.0 min
[B1_f0] Ep 30/30 | loss 0.5207 | val F1 98.48% | best 98.48% (ep 28) | 10.7 min
--> B1 _f0 TEST: Acc=98.389% | Macro-F1=98.312% | Kappa=0.9821 | ECE=0.0849

[B2_f0] Ep 01/30 | loss 2.5201 | val F1 88.20% | best 88.20% (ep 1) | 0.4 min
[B2_f0] Ep 05/30 | loss 0.6322 | val F1 96.25% | best 96.25% (ep 5) | 1.8 min
[B2_f0] Ep 10/30 | loss 0.5666 | val F1 97.44% | best 97.55% (ep 9) | 3.7 min
[B2_f0] Ep 15/30 | loss 0.5452 | val F1 97.92% | best 97.92% (ep 15) | 5.5 min
[B2_f0] Ep 20/30 | loss 0.5310 | val F1 98.00% | best 98.0

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

... (8 outputs left out)

/usr/local/lib/python3.13/dist-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  numerator += self.values[as_s] * \


[B3_f4] Ep 01/30 | loss 0.8272 | val F1 93.13% | best 93.13% (ep 1) | 0.2 min
[B3_f4] Ep 05/30 | loss 0.5711 | val F1 97.37% | best 97.37% (ep 5) | 1.1 min
[B3_f4] Ep 10/30 | loss 0.5419 | val F1 98.14% | best 98.14% (ep 10) | 2.3 min
[B3_f4] Ep 15/30 | loss 0.5150 | val F1 98.27% | best 98.27% (ep 15) | 3.4 min
[B3_f4] Ep 20/30 | loss 0.5066 | val F1 98.28% | best 98.70% (ep 17) | 4.5 min
[B3_f4] Ep 25/30 | loss 0.5035 | val F1 98.89% | best 98.91% (ep 24) | 5.7 min
[B3_f4] Ep 30/30 | loss 0.5031 | val F1 98.91% | best 98.97% (ep 26) | 6.8 min
--> B3 _f4 TEST: Acc=98.907% | Macro-F1=98.851% | Kappa=0.9878 | ECE=0.0819



,Exp,Fold,Acc,Macro_F1,Kappa,ECE,Best_Epoch,Train_Min
0,B1,0,98.389,98.312,0.9821,0.0849,28,10.74
1,B2,0,98.204,98.122,0.9800,0.0807,26,11.05
2,B3,0,98.741,98.677,0.9860,0.0832,26,6.71
3,B1,1,98.370,98.309,0.9819,0.0844,26,10.74
4,B2,1,98.185,98.116,0.9798,0.0864,22,11.16
5,B3,1,98.796,98.748,0.9866,0.0800,25,6.72
6,B1,2,98.796,98.751,0.9866,0.0889,22,10.76
7,B2,2,98.333,98.234,0.9814,0.0867,27,11.14
8,B3,2,98.815,98.767,0.9868,0.0825,29,6.72
9,B1,3,98.444,98.384,0.9827,0.0895,29,10.77


Acc        Macro_F1         Kappa           ECE       
       mean    std     mean    std   mean    std   mean    std
Exp                                                           
B1   98.474  0.183   98.413  0.192  0.983  0.002  0.087  0.002
B2   98.241  0.087   98.157  0.084  0.980  0.001  0.086  0.003
B3   98.818  0.060   98.762  0.062  0.987  0.001  0.082  0.001

## Cell 5 - List the outputs


In [5]:
print("files in", WORK)
for f in sorted(os.listdir(WORK)): print(f"  {f:40s} {os.path.getsize(os.path.join(WORK, f))/1e6:8.2f} MB")


files in /kaggle/working
  __notebook__.ipynb                           0.11 MB
  nb9b_cv_probs.npz                            5.63 MB
  nb9b_cv_results.csv                          0.00 MB
